# Results from the ML model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

In [ ]:
def compute_metrics(true, pred, label):
    rmse = np.sqrt(mean_squared_error(true, pred))
    mae  = mean_absolute_error(true, pred)
    r2   = r2_score(true, pred)
    print(f"Metrics for {label}:")
    print(f"    RMSE = {rmse:.4f} GeV")
    print(f"    MAE  = {mae:.4f} GeV")
    print(f"    R²   = {r2:.4f}")
    return rmse, mae, r2

In [ ]:

def _scatter(x, y, label, color, xlabel, title):
    """Base scatter routine shared by both plot functions."""
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]

    plt.figure(figsize=(10, 8))
    plt.scatter(x, y, s=4, alpha=0.35, color=color, label=label, rasterized=True)

    lim_max = max(x.max(), y.max()) * 1.05
    lim = [0, lim_max]
    plt.plot(lim, lim, "k--", lw=1, label="y = x")

    plt.title(title, fontsize=13, fontweight="bold")
    plt.xlabel(xlabel, fontsize=11)
    plt.ylabel("GenMET_pt [GeV]", fontsize=11)
    plt.legend(markerscale=3, fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    return

In [ ]:
def plot_predicted_MET(df, label, color):
    """
        Scatter of predicted MET (x) vs true GenMET (y).
        Reads from the model output parquet: columns 'y_test_pred' and 'y_test'.
    """
    x = df["y_test_pred"]
    y = df["y_test"]

    print(f"Statistics for dataset {label}:")
    print(f"    Mean predicted MET  : {x.mean():.4f}   Median: {x.median():.4f}")
    print(f"    Mean testing GenMET : {y.mean():.4f}   Median: {y.median():.4f}")

    compute_metrics(y.to_numpy(), x.to_numpy(), label)

    _scatter(
        x=x, y=y,
        label=label,
        color=color,
        xlabel="Predicted MET_pt [GeV]",
        title=f"GenMET vs Predicted MET — {label}",
    )
    return

In [ ]:
def plot_true_MET(df_raw, label, color):
    """
        Scatter of raw reconstructed MET_pt (x) vs GenMET_pt (y).
        Reads from the original dataset parquet: columns 'MET_pt' and 'GenMET_pt'.
    """
    x = pd.Series(df_raw["MET_pt"])
    y = pd.Series(df_raw["GenMET_pt"])

    print(f"Statistics for dataset {label} (raw MET):")
    print(f"    Mean MET_pt   : {x.mean():.4f}   Median: {x.median():.4f}")
    print(f"    Mean GenMET_pt: {y.mean():.4f}   Median: {y.median():.4f}")

    compute_metrics(y.to_numpy(), x.to_numpy(), label + " baseline")

    _scatter(
        x=x, y=y,
        label=label,
        color=color,
        xlabel="Reconstructed MET_pt [GeV]",
        title=f"GenMET vs Reconstructed MET — {label}",
    )

---

## ZZTo2L2Nu

In [ ]:
EVENT      = "ZZTo2L2Nu"
PRED_FILE  = f"Results/METpred_vs_GenMET_{EVENT}.parquet"
RAW_FILE   = f"CleanedDatasets/{EVENT}.parquet"

# Load model predictions (y_test_pred, y_test)
df_pred = pd.read_parquet(PRED_FILE)

# Load original dataset (MET_pt, GenMET_pt — only what we need)
df_raw = pd.read_parquet(RAW_FILE, columns=["MET_pt", "GenMET_pt"])

# Predicted MET vs GenMET
plot_predicted_MET(df_pred, label=EVENT, color="#f0883e")

# True reconstructed MET vs GenMET (baseline)
plot_true_MET(df_raw, label=EVENT, color="#58a6ff")
